In [1]:
from __future__ import annotations
import operator
from pathlib import Path
from typing import List, Literal,TypedDict,Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import  HumanMessage, SystemMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_groq import ChatGroq



## Defining Schemas

#### Task Model

In [2]:
class Task(BaseModel):
    id: int
    title: str
    goal: str = Field(..., description="One sentence describing what the user is able to do after this section")
    bullets: List[str] = Field(...,
                               min_length=3,
                                 max_length=6,
         description="3-6 concrete, non-overlapping subpoints to cover this section")
    target_words: int = Field(..., description="The target number of words for this section is (120-550)")
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False


#### Evidence Item Schema

In [3]:
class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at : Optional[str] = None # keep tavilyt provides, don't rely on it 
    source: Optional[str] = None # keep tavilyt provides, don't rely on it
    snippet: Optional[str] = None # keep tavilyt provides, don't rely on it

#### Evidence Pack Schema

In [4]:
class EvidencePack(BaseModel):
   evidence: List[EvidenceItem] = Field(default_factory=list)

#### Router Decision Schema

In [5]:
class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal['closed_book', 'hybrid', 'open_book']
    queries : List[str] = Field(default_factory=list)

## Defining State

In [6]:
class State(TypedDict):
    topic: str

    # Routing/Research
    mode: str
    need_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]

    # workers
    sections: Annotated[List[tuple[int,str]],operator.add] # taskid,section_md
    final:  str   

In [7]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    
)

## Defining the Router Node

In [8]:
router_system = """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false):
  Evergreen topics where correctness does not depend on recent facts (concepts, fundamentals).
- hybrid (needs_research=true):
  Mostly evergreen but needs up-to-date examples/tools/models to be useful.
- open_book (needs_research=true):
  Mostly volatile: weekly roundups, "this week", "latest", rankings, pricing, policy/regulation.

If needs_research=true:
- Output 3–10 high-signal queries.
- Queries should be scoped and specific (avoid generic queries like just "AI" or "LLM").
- For open_book weekly roundup, include queries that reflect the last 7 days constraint.
"""

def RoterNode(state: State) -> dict:
    topic = state['topic']
    decider = llm.with_structured_output(RouterDecision)
    decision = decider.invoke(
        [
            SystemMessage(content=router_system),
            HumanMessage(content = {'Topic': topic})
        ]
    )
    return{
        "needs_research": decision.needs_research,
        "mode": decision.mode,
        "queries": decision.queries
    }

### Function for conditional edge

In [9]:
def route_next(state: State) ->str:
    return "research" if state['need_research'] else 'orchestrator' # agr research ki zroort ho to reasearch k node pr jao aur agr nhi to orchestrator k node pr